# Location Selection with E-NAUTILUS: Part 1 (with RPM)
_Generation of reference points_

In [1]:
from desdeo.problem import Constant, Problem, Objective, VariableTypeEnum, Constraint, TensorConstant, TensorVariable, ConstraintTypeEnum
import numpy as np
import pandas as pd
from slugify import slugify
# These are to just suppress warnings in the outputs of the example
import warnings
import pickle

warnings.filterwarnings("ignore")

## Model inputs


In [2]:

# Mininum expected attendance to be worth visiting 
min_att = 4

# Cost constants
# The current gas costs ($/gallon)
raw_dollars_per_gallon = 3.00

# The efficiency of the vehicle (miles/gallon)
raw_mpg = 6.0
# How long the event is (hours)
hours_per_event = 4
driver_salary_per_hour = 19
raw_driver_cost_per_trip = hours_per_event * driver_salary_per_hour

# Food desert threshold (minutes)
raw_food_desert_threshold = 15




## Helper functions

In [3]:
def no_nan(val):
    if pd.isna(val):
        return ""
    else:
        return str(val)
    

## Load and process Constants

In [4]:


home = "Ada"
# Read the adjacency matrix (distance in miles)
adjDist = pd.read_csv("data/adjacencyMatrixDist.csv", index_col=0)
dist2home = adjDist.loc[:,[home]].rename(columns={home:"dist2home"})
dist2home.index.name = "city"
display("Distance to home base")
display(dist2home)

# Read the adjacency matrix (travel time in minutes)
adjTTime = pd.read_csv("data/adjacencyMatrixTravelTime.csv", index_col=0)
display("Travel time adjacency matrix")
display(adjTTime)

# Read the cities 
cities = pd.read_csv("data/cities.csv")

# Data integrity check
cities_cities_df = set(cities.loc[:,"city"])
cities_adjDist_df = set(adjDist.index)
if cities_cities_df != cities_adjDist_df: 
    raise ValueError(f"The city table and the adjacency matrix have differing cities.\nCity table: {cities_cities_df}\nAdjacency matrix{cities_adjDist_df}")


'Distance to home base'

,dist2home
city,
Ada,0.00
Alger,8.81
Bluffton,19.50
Cairo,31.26
Caledonia,102.21
Carey,66.05
Columbus Grove,36.16
Continental,70.66
Cridersville,38.76


'Travel time adjacency matrix'

,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,Cridersville,Delphos,...,Ottawa,Ottoville,Pandora,Prospect,Saint Marys,Spencerville,Sycamore,Upper Sandusky,Wapakoneta,Waynesfield
Ada,0.00,9.30,20.48,23.63,70.40,45.55,31.78,62.73,35.73,35.74,...,41.59,42.32,31.00,72.88,51.50,50.39,51.19,38.63,39.70,32.14
Alger,9.30,0.00,29.61,32.76,79.53,54.68,40.91,71.86,36.43,44.88,...,50.72,51.45,37.84,73.76,52.20,51.39,60.32,47.76,40.39,22.95
Bluffton,20.48,29.61,0.00,18.42,76.23,36.78,20.55,49.98,29.81,30.53,...,26.52,37.11,15.30,84.11,45.58,45.18,56.07,44.46,33.78,37.66
Cairo,23.63,32.76,18.42,0.00,78.76,51.34,11.29,42.34,26.10,17.40,...,19.81,23.98,20.73,86.64,43.96,32.74,59.54,46.98,32.15,37.08
Caledonia,70.40,79.53,76.23,78.76,0.00,47.25,87.77,118.71,95.99,91.73,...,96.46,98.31,86.98,28.59,111.76,107.07,49.45,34.24,99.96,90.16
Carey,45.55,54.68,36.78,51.34,47.25,0.00,57.96,78.84,63.76,64.48,...,55.38,71.06,48.79,53.68,79.53,79.13,19.29,15.77,67.73,71.61
Columbus Grove,31.78,40.91,20.55,11.29,87.77,57.96,0.00,32.45,35.78,27.08,...,9.92,28.86,9.43,94.52,53.64,42.42,67.42,54.87,41.84,46.76
Continental,62.73,71.86,49.98,42.34,118.71,78.84,32.45,0.00,63.28,36.13,...,23.51,26.32,41.28,126.22,74.44,52.23,97.36,86.57,69.34,74.72
Cridersville,35.73,36.43,29.81,26.10,95.99,63.76,35.78,63.28,0.00,36.77,...,44.46,43.35,35.31,101.02,28.29,30.89,74.39,61.84,16.48,27.29
Delphos,35.74,44.88,30.53,17.40,91.73,64.48,27.08,36.13,36.77,0.00,...,35.47,10.96,36.38,99.67,39.11,16.90,72.58,60.02,39.91,48.70


### Site table 

In [5]:
sites_only = pd.read_csv("data/proposedSitesProcessed.csv", index_col=0)

# Data integrity check
cities_cities_df = set(cities.loc[:,"city"])
cities_adjDist_df = set(sites_only.loc[:,"city"])
if len(cities_adjDist_df - cities_cities_df) != 0: 
    print(cities_adjDist_df - cities_cities_df)
    raise ValueError(f"Some cities in the site table don't appear in the city table.\nCity table: {cities_cities_df}\nSite table{cities_adjDist_df}")

# Create event table
sites = pd.merge(sites_only,cities,  on="city")
display(sites)
sites.loc[:, "site_id"] = sites.apply(lambda row: slugify(f'{row["city"]} {row["siteName"]}'), axis=1)
sites.loc[:, "site_pretty"] = sites.apply(lambda row: f'{row["siteName"]}', axis=1)

# Add the distance to home for each site 
sites = pd.merge(sites, dist2home, on="city")

display(sites)

,siteName,siteType,city,minAtt,medAtt,maxAtt,lat,long,pop,svi
0,Alger Dinner,charitable meal,Alger,1.0,3.0,5.0,40.709722,-83.844167,837,0.594200
1,DJ Kirk,charitable meal,Dunkirk,3.0,3.0,3.0,40.788056,-83.642778,774,0.547500
2,DJ Kirk,charitable meal,Dunkirk,2.0,2.0,2.0,40.788056,-83.642778,774,0.547500
3,Dunkirk Dinner,charitable meal,Dunkirk,3.0,3.0,7.0,40.788056,-83.642778,774,0.547500
4,Lima ODB,charitable meal,Lima,2.0,3.0,4.0,40.746389,-84.123333,35579,0.923333
5,Saint Marks UMC,church,Lima,2.0,3.0,7.0,40.746389,-84.123333,35579,0.923333
6,Wapak UCC,church,Wapakoneta,6.0,6.0,6.0,40.565556,-84.191111,9957,0.472500
7,Grace Clinics,clinic,Marion,0.0,3.0,10.0,40.620000,-83.126389,35999,0.932500
8,Christian Corner,food bank,Lima,0.0,6.0,10.0,40.746389,-84.123333,35579,0.923333
9,Alger Food Pantry,food pantry,Alger,2.0,4.0,6.0,40.709722,-83.844167,837,0.594200


,siteName,siteType,city,minAtt,medAtt,maxAtt,lat,long,pop,svi,site_id,site_pretty,dist2home
0,Alger Dinner,charitable meal,Alger,1.0,3.0,5.0,40.709722,-83.844167,837,0.594200,alger-alger-dinner,Alger Dinner,8.81
1,DJ Kirk,charitable meal,Dunkirk,3.0,3.0,3.0,40.788056,-83.642778,774,0.547500,dunkirk-dj-kirk,DJ Kirk,17.85
2,DJ Kirk,charitable meal,Dunkirk,2.0,2.0,2.0,40.788056,-83.642778,774,0.547500,dunkirk-dj-kirk,DJ Kirk,17.85
3,Dunkirk Dinner,charitable meal,Dunkirk,3.0,3.0,7.0,40.788056,-83.642778,774,0.547500,dunkirk-dunkirk-dinner,Dunkirk Dinner,17.85
4,Lima ODB,charitable meal,Lima,2.0,3.0,4.0,40.746389,-84.123333,35579,0.923333,lima-lima-odb,Lima ODB,26.28
5,Saint Marks UMC,church,Lima,2.0,3.0,7.0,40.746389,-84.123333,35579,0.923333,lima-saint-marks-umc,Saint Marks UMC,26.28
6,Wapak UCC,church,Wapakoneta,6.0,6.0,6.0,40.565556,-84.191111,9957,0.472500,wapakoneta-wapak-ucc,Wapak UCC,50.39
7,Grace Clinics,clinic,Marion,0.0,3.0,10.0,40.620000,-83.126389,35999,0.932500,marion-grace-clinics,Grace Clinics,91.69
8,Christian Corner,food bank,Lima,0.0,6.0,10.0,40.746389,-84.123333,35579,0.923333,lima-christian-corner,Christian Corner,26.28
9,Alger Food Pantry,food pantry,Alger,2.0,4.0,6.0,40.709722,-83.844167,837,0.594200,alger-alger-food-pantry,Alger Food Pantry,8.81


### Close cities for events

In [6]:

close_cities = adjTTime < raw_food_desert_threshold
cities_adj2sites = sites.loc[:,["city", "site_id"]].merge(close_cities, left_on="city", right_index=True)
display(cities_adj2sites)
cities_adj2sites = (cities_adj2sites.iloc[:,2:].values).astype(int)
s_adj_raw = cities_adj2sites.T
display(cities_adj2sites.T.shape)
e_adj_raw_list = s_adj_raw.tolist()

,city,site_id,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,...,Ottawa,Ottoville,Pandora,Prospect,Saint Marys,Spencerville,Sycamore,Upper Sandusky,Wapakoneta,Waynesfield
0,Alger,alger-alger-dinner,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,Dunkirk,dunkirk-dj-kirk,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,Dunkirk,dunkirk-dj-kirk,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,Dunkirk,dunkirk-dunkirk-dinner,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,Lima,lima-lima-odb,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,Lima,lima-saint-marks-umc,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
6,Wapakoneta,wapakoneta-wapak-ucc,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
7,Marion,marion-grace-clinics,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
8,Lima,lima-christian-corner,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9,Alger,alger-alger-food-pantry,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


(36, 60)

In [7]:
site_dist2home = sites.loc[:,["dist2home"]].T.values
site_dist2home_list = site_dist2home.tolist()

## Constants

In [8]:
site_count = sites.shape[0]

# Expected attenance 
raw_attendance = sites.loc[:,["medAtt"]] 
raw_under_attendance = (raw_attendance < min_att).astype(int)

# Site SVI
raw_svi = sites.loc[:,["svi"]]

svi = TensorConstant(name="Site SVI",
                     symbol="svi", 
                     shape=raw_svi.T.shape,
                     values=raw_svi.T.values.tolist())
display(svi)

# How many people to expect to attend each site
ea = TensorConstant(name="Expected attendance",
                                 symbol="ea", 
                                 shape=raw_attendance.T.shape,
                                 values=raw_attendance.T.values.tolist())
display(ea)

# How many sites are expected to be under attended? 
eua = TensorConstant(name="Expected under-attendance",
                    symbol="eua",
                    shape=raw_under_attendance.T.shape,
                    values=raw_under_attendance.T.values.tolist())

display(eua)

# Distance to home 
sd2h = TensorConstant(name="The distance to drive from home to a site",
                    symbol="sd2h",                        
                    shape=site_dist2home.shape,
                    values=site_dist2home_list)
display(sd2h)

mpg = Constant(name="Miles per gallon (miles/gallon)", 
               symbol="mpg", 
               value=raw_mpg)

display(mpg)
dpg = Constant(name="Dollars per gallon ($/gallon)", 
               symbol="dpg", 
               value=raw_dollars_per_gallon)

display(dpg)
dcpt = Constant(name="Driver cost per trip ($)", 
                                symbol="dcpt", 
                                value=raw_driver_cost_per_trip)

mpg_inv_times_dpg = Constant(name="(1/(miles per gallon)) x dollars per gallon", 
                                symbol="mpg_inv_times_dpg", 
                                value=(1/raw_mpg)*raw_dollars_per_gallon)


display(dcpt)

# Site adjacency to cities
s_adj = TensorConstant(name="What cities are adjacent to a given site",
                      symbol="s_adj",                      
                      shape=s_adj_raw.shape,
                      values=e_adj_raw_list)
display(s_adj)

city_pops = TensorConstant(name="City populations", 
                           symbol="cpop", 
                           shape=[cities.shape[0],1],
                           values=cities.loc[:,["pop"]].values.tolist()
                           )

display(city_pops)

total_pop = Constant(name="Total population of interest", 
                     symbol="tpop", 
                     value=float(sum(cities.loc[:,"pop"])))

display(total_pop)


TensorConstant(name='Site SVI', symbol='svi', shape=[1, 60], values=['List', ['List', 0.5942, 0.5475, 0.5475, 0.5475, 0.923333333, 0.923333333, 0.4725, 0.9325, 0.923333333, 0.5942, 0.4375, 0.4375, 0.9183, 0.9183, 0.9183, 0.4508, 0.923333333, 0.39, 0.5942, 0.19, 0.3275, 0.4142, 0.3508, 0.3992, 0.5625, 0.565, 0.1958, 0.1, 0.0675, 0.7117, 0.66, 0.3217, 0.0625, 0.1058, 0.4375, 0.6767, 0.5475, 0.16625, 0.9183, 0.1633, 0.7117, 0.923333333, 0.9325, 0.2633, 0.3483, 0.095, 0.2467, 0.0967, 0.5517, 0.195, 0.6392, 0.4725, 0.3467, 0.5625, 0.9183, 0.9183, 0.9183, 0.9183, 0.923333333, 0.923333333]])

TensorConstant(name='Expected attendance', symbol='ea', shape=[1, 60], values=['List', ['List', 3.0, 3.0, 2.0, 3.0, 3.0, 3.0, 6.0, 3.0, 6.0, 4.0, 0.0, 2.0, 3.0, 3.0, 2.0, 7.0, 3.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 7.0, 3.0, 2.0, 4.0, 6.0, 1.0, 3.0]])

TensorConstant(name='Expected under-attendance', symbol='eua', shape=[1, 60], values=['List', ['List', 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1]])

TensorConstant(name='The distance to drive from home to a site', symbol='sd2h', shape=[1, 60], values=['List', ['List', 8.81, 17.85, 17.85, 17.85, 26.28, 26.28, 50.39, 91.69, 26.28, 8.81, 69.13, 69.13, 24.99, 24.99, 24.99, 19.5, 26.28, 0.0, 8.81, 31.26, 102.21, 66.05, 36.16, 70.66, 51.03, 44.79, 35.97, 57.86, 52.05, 48.76, 39.74, 47.6, 60.42, 35.75, 69.13, 38.76, 17.85, 50.5, 24.99, 52.09, 48.76, 26.28, 91.69, 84.84, 71.7, 79.5, 69.38, 78.84, 49.06, 68.87, 53.51, 50.39, 34.99, 51.03, 24.99, 24.99, 24.99, 24.99, 26.28, 26.28]])

Constant(name='Miles per gallon (miles/gallon)', symbol='mpg', value=6.0)

Constant(name='Dollars per gallon ($/gallon)', symbol='dpg', value=3.0)

Constant(name='Driver cost per trip ($)', symbol='dcpt', value=76)

TensorConstant(name='What cities are adjacent to a given site', symbol='s_adj', shape=[36, 60], values=['List', ['List', 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1], ['List', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

TensorConstant(name='City populations', symbol='cpop', shape=[36, 1], values=['List', ['List', 5334], ['List', 837], ['List', 3967], ['List', 517], ['List', 560], ['List', 3565], ['List', 2160], ['List', 1102], ['List', 1791], ['List', 7117], ['List', 774], ['List', 1923], ['List', 1350], ['List', 525], ['List', 969], ['List', 1455], ['List', 7947], ['List', 676], ['List', 2177], ['List', 35579], ['List', 35999], ['List', 3046], ['List', 601], ['List', 706], ['List', 3034], ['List', 946], ['List', 4456], ['List', 966], ['List', 1204], ['List', 1067], ['List', 8397], ['List', 2198], ['List', 793], ['List', 6698], ['List', 9957], ['List', 749]])

Constant(name='Total population of interest', symbol='tpop', value=161142.0)

## Variables

In [9]:
sv = TensorVariable(
  name="Sites visited",                  
  symbol="sv",
  variable_type=VariableTypeEnum.integer,
  shape=[sites.shape[0],1],
  lowerbounds=0,
  upperbounds=1,
  initial_values=0)

display(sv)

cover = TensorVariable(
    name="Coverage of cities",
    symbol="cover", 
    variable_type=VariableTypeEnum.integer,
    shape=[adjTTime.shape[0],1],
    lowerbounds=0,
    upperbounds=1,
    initial_values=0)

cover



TensorVariable(name='Sites visited', symbol='sv', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[60, 1], lowerbounds=0, upperbounds=1, initial_values=0)

TensorVariable(name='Coverage of cities', symbol='cover', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[36, 1], lowerbounds=0, upperbounds=1, initial_values=0)

## Constraints

Number of sites is $s$, number of cities is $c$

`sv` = 
$\begin{bmatrix}
x_1\\
x_2\\
⋮ \\
x_s\\
\end{bmatrix}$

- determines whether or not a given site is visited. 
- a binary vector of size $s$ 
- $s$ is the number of sites under consideration


`cover` = 
$\begin{bmatrix}
x^{'}_1\\
x^{'}_2\\
⋮ \\
x^{'}_c\\
\end{bmatrix}$

- A support variable that represents whether a city is covered by the given subset of selected sites in `sv.
- A binary vector of size $c$
- $c$ is the number of cities under consideration

`s_adj` = 
$\begin{bmatrix}
d_{11},…,d_{1s}   \\
⋮,⋱,⋮,\\
d_{c1},…,d_{cs}   \\
\end{bmatrix}$

- A constant that represents whether or not a given site (in the columns) is adjacent to a give city (in the rows)




In [10]:
# sv = [18, 1]
# cover [28, 1]
# s_adj [28, 18]
# [28, 18] . [18, 1] = [28, 1]


g = Constraint(
      name="Desert constraint",
      symbol="c",
      func="cover-s_adj@sv",
      cons_type=ConstraintTypeEnum.LTE,
      is_convex=False,
      is_linear=True,
      is_twice_differentiable=True)

## Objectives

In [11]:

# REMEMBER! If you change the objective, you may need to change the table formatting in step02
# Total patients seen 
total_patients = Objective(
    name = "Maximize total patients served",
    symbol = "f_1", 
    maximize = True,
    is_twice_differentiable=True,
    func = "Sum(ea@sv)"                                    
)

# Underattended sites
under_attended_sites = Objective(
    name = "Minimize the number sites taht are under-attended", 
    symbol = "f_2",
    maximize = False,
    is_twice_differentiable=True,
    func = "Sum(eua@sv)"                                
)

# Cost of events
costs = Objective(
    name = "Minimize total costs ($)",
    symbol = "f_3",
    maximize = False,
    is_twice_differentiable=True,
    func = "Sum(sd2h@sv)*mpg_inv_times_dpg + Sum(sv)*dcpt"    
)

# cpop: [1, 28]
# cover: [28, 1]
coverage = Objective(
    name = "Maximize population with clinic access", 
    symbol = "f_4", 
    maximize = True, 
    is_twice_differentiable=True,
    func = "Sum(cover*cpop)"
)

cumul_svi = Objective(
    name = "Cumulative SVI", 
    symbol = "f_5", 
    maximize = True, 
    is_twice_differentiable=True, 
    func = "Sum(svi@sv)"
)


## Problem

In [12]:
prob = Problem(
        name="Simple site selection",
        description="Simple implementation of the site selection problem",
        is_linear=True,
        is_convex=False,
        is_twice_differentiable=True,
        constraints=[g],
        constants=[eua, ea, sd2h, mpg, dpg, dcpt, s_adj, city_pops, total_pop, mpg_inv_times_dpg, svi],
        variables=[sv, cover],
        objectives=[total_patients, under_attended_sites, costs, coverage, cumul_svi]
    )

## Ideal/nadir

In [14]:

# f_1 Total patients seen 
# f_2 Underattended sites
# f_3 Cost of events
# f_4 Number of citizens covered

# f_1 ideal is seeing all patients, nadir is seeing no patients
all_patients = int(np.sum(sites.loc[:,"medAtt"]))

# f_2 ideal is having no under-attended sites
# f_2 naird is having visiting all locations with under-attended sites
all_ose = int(np.sum(raw_under_attendance))


# How many miles are driven/gas costs
max_dist = sites.loc[:,"dist2home"].sum()
total_gas_cost = (max_dist / raw_mpg) * raw_dollars_per_gallon
driver_cost = sites.shape[0]*raw_driver_cost_per_trip
max_costs = float(driver_cost + total_gas_cost)

# Total number of people
total_citizens = int(cities.loc[:,["pop"]].sum().values[0])

# Maximum SVI score
max_cumul_svi = float(np.sum(raw_svi))

prob = prob.update_ideal_and_nadir(
    new_ideal={
        "f_1": all_patients,
        "f_2": 0, 
        "f_3": 0, 
        "f_4": total_citizens,
        "f_5": max_cumul_svi
        }, 
    new_nadir={
        "f_1": 0,
        "f_2": all_ose,
        "f_3": max_costs, 
        "f_4": 0,
        "f_5": 0
        }
    )


print(f"Ideal values: {prob.get_ideal_point()}")
print(f"Nadir values: {prob.get_nadir_point()}")

Ideal values: {'f_1': 298, 'f_2': 0, 'f_3': 0, 'f_4': 161142, 'f_5': 32.842883330999996}
Nadir values: {'f_1': 0, 'f_2': 17, 'f_3': 5862.45, 'f_4': 0, 'f_5': 0}


## RPM solver


In [16]:
from desdeo.mcdm.reference_point_method import rpm_solve_solutions
from itertools import product
from desdeo.tools import PyomoBonminSolver, PyomoCBCSolver, PyomoGurobiSolver, PyomoIpoptSolver
from desdeo.tools import NevergradGenericSolver

ref_resolution = 3

f_1_ref = np.linspace(prob.get_ideal_point()["f_1"], prob.get_nadir_point()["f_1"],ref_resolution).tolist()
f_2_ref = np.linspace(prob.get_ideal_point()["f_2"], prob.get_nadir_point()["f_2"],ref_resolution).tolist()
f_3_ref = np.linspace(prob.get_ideal_point()["f_3"], prob.get_nadir_point()["f_3"],ref_resolution).tolist()
f_4_ref = np.linspace(prob.get_ideal_point()["f_4"], prob.get_nadir_point()["f_4"],ref_resolution).tolist()
f_5_ref = np.linspace(prob.get_ideal_point()["f_5"], prob.get_nadir_point()["f_5"],ref_resolution).tolist()
#f_1_ref = [200]
#f_2_ref = [10]
#f_3_ref = [1000]
#f_4_ref = [0.5]

pf_samples_raw = []
i = 0 
for ref in product(f_1_ref, f_2_ref, f_3_ref, f_4_ref, f_5_ref):
    reference_point = {"f_1": ref[0], "f_2": ref[1], "f_3": ref[2], "f_4": ref[3], "f_5": ref[4]}

    print(f"Calculating for ref #{i} {reference_point}")
    try: 
        res = rpm_solve_solutions(prob, reference_point=reference_point, solver=PyomoGurobiSolver)
        pf_samples_raw.append(res)
    except ValueError:
        print("Error running for that ref.")
    i += 1


Calculating for ref #0 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 161142.0, 'f_5': 32.842883330999996}
Calculating for ref #1 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 161142.0, 'f_5': 16.421441665499998}
Calculating for ref #2 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 161142.0, 'f_5': 0.0}
Calculating for ref #3 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 80571.0, 'f_5': 32.842883330999996}
Calculating for ref #4 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 80571.0, 'f_5': 16.421441665499998}
Calculating for ref #5 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 80571.0, 'f_5': 0.0}
Calculating for ref #6 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 0.0, 'f_5': 32.842883330999996}
Calculating for ref #7 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 0.0, 'f_5': 16.421441665499998}
Calculating for ref #8 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 0.0, 'f_5': 0.0}
Calculating for ref #9 {'f_1': 298.0, 'f_2': 0.0, 'f_3': 2931.225, 'f_4': 161142.0, 'f_5': 32.842883330999996

## Send relevant data to a pkl file

In [1]:
import pickle
output = open(f'data/pf_test.pkl', 'wb')
pickle.dump({"pf": pf_samples_raw,
             "prob" : prob,
             "cities": cities,
             "sites": sites,
             "cities_adj2sites": cities_adj2sites, 
             "total_pop": total_citizens
             }, output)


NameError: name 'pf_samples_raw' is not defined